In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

# Sentiment Analysis Pipeline

In [ ]:
sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

In [ ]:
result = sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
# Specifying Sentiment Analyis Model
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

# Named Entity Recognition

In [ ]:
named_entity_recognition = pipeline("ner", device="cuda")
result = named_entity_recognition("Sundar Pichai is the CEO of Google. He was born in India and now works in California.")

for entity in result:
    print(entity)


# Question Answering with Context


In [ ]:
context = """
Python is a high-level programming language created by Guido van Rossum.
It was first released in 1991.
Python is widely used for web development, machine learning, data science,
automation, and artificial intelligence.
"""

question_answer = pipeline("question-answering", device="cuda")
result = question_answer(question="Who created python?", context=context)
print(result)

# Text Summarization

In [ ]:
summarizer = pipeline("summarization", device="cuda")

text = """
Artificial Intelligence (AI) is transforming industries across the world.
It is being used in healthcare to assist doctors in diagnosing diseases,
in finance to detect fraud, in education to personalize learning,
and in transportation to develop self-driving cars.
As AI continues to evolve, concerns about ethics, privacy,
and job displacement are also increasing. Researchers and policymakers
are working together to ensure that AI is developed responsibly.
"""

summary = summarizer(text, max_length=50, min_length=10, do_sample=False)
print(summary)

# Translation


In [ ]:
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", device="cuda")
result = translator("Artificial Intelligence is transforming the world.", src_lang="eng_Latn", tgt_lang="mar_Deva")
print(result)

# Classification


In [ ]:
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Artificial Intelligence is transforming healthcare.", candidate_labels=["technology","sports","politics","Health"])
print(result)

# Text Generation

In [ ]:
generator = pipeline("text-generation", device="cuda")
result = generator("Once upon a time, ")
print(result[0]['generated_text'])

# Image Generation
# Pipelines can be used for diffusion models as well as transformers

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

In [ ]:
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of cats learning AI Engineering"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

# Audio Generation

In [ ]:
from transformers import pipeline
from IPython.display import Audio

tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-eng",
    device="cuda"
)

speech = tts("Hello, how are you?")

Audio(speech["audio"], rate=speech["sampling_rate"])